In [1]:
import logging

from beir import LoggingHandler
from beir.datasets.data_loader import GenericDataLoader
from beir.retrieval.evaluation import EvaluateRetrieval
from beir.retrieval.search.dense import DenseRetrievalExactSearch

import numpy as np
import pandas as pd

from stemmer import Stemmer, tokenize
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

/home/fahmi/freelance/project-2023-amsearch/.venv/lib/python3.11/site-packages/beir/datasets/data_loader.py:2: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


In [2]:
logging.basicConfig(
    format="%(asctime)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    level=logging.INFO,
    handlers=[LoggingHandler()],
)

## Helpers

In [3]:
class AMSTokenizer:
    def __init__(self, stm: Stemmer):
        self.stemmer = stm

    def __call__(self, doc):
        return [self.stemmer.stem_ams(word) for word in tokenize(doc)]

In [4]:
stemmer = Stemmer("../data/sundabaru1-vocab.txt")
amstokenizer = AMSTokenizer(stemmer)

In [5]:
df_corpus = pd.read_json("../data/triplet/triplet.jsonl", lines=True)
df_corpus.head()

,query,positive,negative
0,Kumaha cara ngahontal kahayang dina kahirupan?,"Kahiji, urang kedah sabar sareng henteu janten...",Abdi ngadangu seueur warta ngeunaan jalma anu ...
1,Naon anu kedah dilakukeun lamun gering?,"Lamun gering, ulah rungsing sabab pikiran posi...",Masyarakat ayeuna seueur nganggur di kota nu g...
2,Kumaha cara nyieun amal?,"Ngawitan amal ti hal-hal leutik, sapertos ngab...",Jalma sering nyarita ngeunaan kaékonomian anu ...
3,Kumaha sangkan ngeterkeun diri ka Gusti?,"Mertahankeun ati, pariksa tindakan sorangan, s...","Saurang guru ngajarkeun pentingna ilmu, tapi k..."
4,Naon hartina jadi jalma leutik?,Jadi jalma leutik hartina ulah sombong sareng ...,"Dina pagelaran, anu katinggali gaduh prestasi ..."


In [6]:
train_corpus = df_corpus.values.ravel().tolist()
len(train_corpus), train_corpus[0]

(22476, 'Kumaha cara ngahontal kahayang dina kahirupan?')

## Bag-of-words / TF-IDF

In [7]:
model_name = "ams-bag_of_words"
eval_stemming = True

# vsm_model = CountVectorizer()
vsm_model = CountVectorizer(tokenizer=amstokenizer)
# vsm_model = TfidfVectorizer()
# vsm_model = TfidfVectorizer(tokenizer=amstokenizer)

vsm_model.fit(train_corpus)

/home/fahmi/freelance/project-2023-amsearch/.venv/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


CountVectorizer(tokenizer=<__main__.AMSTokenizer object at 0x7f3180802ed0>)

In [8]:
vsm_model.tokenizer

## Evaluation

In [17]:
def stem_sentence(text: str) -> str:
    return " ".join([stemmer.stem_ams(t) for t in tokenize(text)])

def stem_corpus(item: dict[str, str]):
    return {k: stem_sentence(v) for k, v in item.items()}

In [19]:
corpus, queries, qrels = GenericDataLoader(data_folder="../data/beir", qrels_file=f"../data/beir/qrels.tsv").load_custom()
if eval_stemming:
    queries = stem_corpus(queries)
    corpus = {k: stem_corpus(v) for k, v in corpus.items()}

2025-04-05 13:30:46 - Loading Corpus...


100%|██████████| 1499/1499 [00:00<00:00, 107921.00it/s]

2025-04-05 13:30:46 - Loaded 1499 Documents.
2025-04-05 13:30:46 - Doc Example: {'text': 'bismillah yuga lampah balukar janglar meunang kahayang balukar sasar pedar ringkang sing jembar sabar tong jadi hambar ihtér raksa rasa ukir pikir mun gering tong rungsing mun cageur tong badeur ulah ceurik jadi jalma leutik ulah sombong abong di gedong lain batur kabéh gelé dulur néang halal awur amal ulah lieur ku madu dunya jung sanding ka nu agung gusti wéti ngarti nu sajati pariksa ati', 'title': 'JEMBAR SABAR'}
2025-04-05 13:30:46 - Loading Queries...
2025-04-05 13:30:46 - Loaded 7491 Queries.
2025-04-05 13:30:46 - Query Example: apa maksud dari bismillah yuga lampah


In [20]:
# https://github.com/beir-cellar/beir/wiki/Evaluate-your-custom-model
class BaselineModel:
    def __init__(self, model=None, **kwargs):
        self.model = model
    
    # Write your own encoding query function (Returns: Query embeddings as numpy array)
    def encode_queries(self, queries: list[str], batch_size: int, **kwargs) -> np.ndarray:
        return self.model.transform(queries).todense().astype(float)
    
    # Write your own encoding corpus function (Returns: Document embeddings as numpy array)  
    def encode_corpus(self, corpus: list[dict[str, str]], batch_size: int, **kwargs) -> np.ndarray:
        extracted_corpus = [row["title"] + " " + row["text"] for row in corpus]
        return self.model.transform(extracted_corpus).todense().astype(float)

dres_model = DenseRetrievalExactSearch(BaselineModel(vsm_model), batch_size=16)
dres_model

In [21]:
retriever = EvaluateRetrieval(dres_model, score_function="cos_sim")  # dot or cos_sim
results = retriever.retrieve(corpus, queries)

2025-04-05 13:30:50 - Encoding Queries...
2025-04-05 13:30:52 - Sorting Corpus by document length (Longest first)...
2025-04-05 13:30:52 - Encoding Corpus in batches... Warning: This might take a while!
2025-04-05 13:30:52 - Scoring Function: Cosine Similarity (cos_sim)
2025-04-05 13:30:52 - Encoding Batch 1/1...


In [ ]:
#### Evaluate your model with NDCG@k, MAP@K, Recall@K and Precision@K  where k = [1,3,5,10,100,1000]
ndcg, _map, recall, precision = retriever.evaluate(qrels, results, retriever.k_values)
metrics = [
    {
        "model": model_name, 
        "stemming": eval_stemming,
        "metric": k.split("@")[0], 
        "k": k.split("@")[1], 
        "value": v
    } for col in [ndcg, _map, recall, precision] for k, v in col.items()
]

df_metrics = pd.DataFrame(metrics)
df_metrics.to_csv(f"eval-{model_name}-{eval_stemming}.csv", index=None)

2025-04-05 13:30:10 - For evaluation, we ignore identical query and document ids (default), please explicitly set ``ignore_identical_ids=False`` to ignore this.
2025-04-05 13:30:13 - 

2025-04-05 13:30:13 - NDCG@1: 0.2383
2025-04-05 13:30:13 - NDCG@3: 0.3194
2025-04-05 13:30:13 - NDCG@5: 0.3454
2025-04-05 13:30:13 - NDCG@10: 0.3713
2025-04-05 13:30:13 - NDCG@100: 0.4206
2025-04-05 13:30:13 - NDCG@1000: 0.4476
2025-04-05 13:30:13 - 

2025-04-05 13:30:13 - MAP@1: 0.2383
2025-04-05 13:30:13 - MAP@3: 0.2995
2025-04-05 13:30:13 - MAP@5: 0.3139
2025-04-05 13:30:13 - MAP@10: 0.3247
2025-04-05 13:30:13 - MAP@100: 0.3342
2025-04-05 13:30:13 - MAP@1000: 0.3351
2025-04-05 13:30:13 - 

2025-04-05 13:30:13 - Recall@1: 0.2383
2025-04-05 13:30:13 - Recall@3: 0.3769
2025-04-05 13:30:13 - Recall@5: 0.4400
2025-04-05 13:30:13 - Recall@10: 0.5198
2025-04-05 13:30:13 - Recall@100: 0.7580
2025-04-05 13:30:13 - Recall@1000: 0.9772
2025-04-05 13:30:13 - 

2025-04-05 13:30:13 - P@1: 0.2383
2025-04-05 13:30:13